# Assignment 05 — Feature Engineering & Regression Modeling

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    r2_score, mean_absolute_error, mean_squared_error,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, roc_auc_score, ConfusionMatrixDisplay
)
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
RANDOM_STATE = 42

 Part A — Data Cleaning & Preparation

In [ ]:
df = pd.read_csv('student_dataset_dirty.csv')
print("Shape:", df.shape)
df.head()
df.info()
df.describe(include='all')
# Missing values per column
missing = df.isna().sum().to_frame('missing_count')
missing['missing_pct'] = (missing['missing_count'] / len(df) * 100).round(2)
# Duplicate rows
print("Fully duplicated rows:", df.duplicated().sum())
print("Duplicated Student_IDs:", df['Student_ID'].duplicated().sum())
# Inconsistent categorical labels — inspect raw unique values
cat_cols_to_check = ['Gender', 'City', 'Class', 'Parent_Education',
                      'Internet_Access', 'Result']
for c in cat_cols_to_check:
   print(sorted(df[c].dropna().unique(), key=str))
   print()
# Invalid / outlier values in numeric-ish columns (currently stored as strings
# because of stray text like 'absent', 'error', 'N/A', spelled-out numbers, etc.)
numeric_like_cols = ['Age', 'Attendance_Percentage', 'Study_Hours_per_day',
                      'Math_Score', 'Science_Score', 'English_Score', 'Total_Score']
for c in numeric_like_cols:
    non_numeric = df[c][pd.to_numeric(df[c], errors='coerce').isna() & df[c].notna()]
    print(f"{c}: dtype={df[c].dtype}, non-numeric distinct values -> "
          f"{sorted(non_numeric.unique(), key=str)}")
before = df.shape[0]
df = df.drop_duplicates().reset_index(drop=True)
after = df.shape[0]
print(f"Removed {before - after} fully duplicated rows. New shape: {df.shape}")
df['Student_ID'] = df['Student_ID'].astype(str).str.strip()
df['Student_ID'].head(10)
def normalize_text(s):
    if pd.isna(s):
        return np.nan
    return str(s).strip().lower()

df['Gender'] = df['Gender'].apply(normalize_text)
gender_map = {
    'm': 'Male', 'male': 'Male',
    'f': 'Female', 'female': 'Female',
    'o': 'Other', 'other': 'Other',
}
df['Gender'] = df['Gender'].map(gender_map)
print(df['Gender'].value_counts(dropna=False))
# fix casing + common typos/abbreviations
df['City'] = df['City'].apply(normalize_text)
city_map = {
    'mymensign': 'mymensingh', 'mymensingh': 'mymensingh',
    'dhaka': 'dhaka', 'dacca': 'dhaka',
    'khulna': 'khulna',
    'rajshahi': 'rajshahi',
    'sylhet': 'sylhet',
    'barisal': 'barisal', 'barishal': 'barisal',
    'rangpur': 'rangpur',
    'chattogram': 'chattogram', 'chittagong': 'chattogram', 'ctg': 'chattogram',
}
df['City'] = df['City'].map(city_map)
df['City'] = df['City'].str.title()
print(df['City'].value_counts(dropna=False))
# unify word / ordinal / numeral forms into a single grade number 6-10
df['Class'] = df['Class'].apply(normalize_text)
class_map = {
    'six': 6, '6th': 6, '6': 6,
    'seven': 7, '7th': 7, '7': 7,
    'eight': 8, '8th': 8, '8': 8,
    'nine': 9, '9th': 9, '9': 9,
    'ten': 10, '10th': 10, '10': 10,
}
df['Class'] = df['Class'].map(class_map)
print(df['Class'].value_counts(dropna=False).sort_index())
# Parent_Education
df['Parent_Education'] = df['Parent_Education'].apply(normalize_text)
edu_map = {
    'illiterate': 'Illiterate',
    'primary': 'Primary',
    'secondary': 'Secondary',
    'higher secondary': 'Higher Secondary',
    'graduate': 'Graduate',
    'post graduate': 'Post Graduate',
    'n/a': np.nan,   # explicit "not applicable" entered as text
}
df['Parent_Education'] = df['Parent_Education'].map(edu_map)
print(df['Parent_Education'].value_counts(dropna=False))
# unify yes/no encodings: yes/no, y/n, 1/0
df['Internet_Access'] = df['Internet_Access'].apply(normalize_text)
internet_map = {
    'yes': 'Yes', 'y': 'Yes', '1': 'Yes',
    'no': 'No', 'n': 'No', '0': 'No',
}
df['Internet_Access'] = df['Internet_Access'].map(internet_map)
print(df['Internet_Access'].value_counts(dropna=False))

df['Result'] = df['Result'].apply(normalize_text)
result_map = {
    'pass': 'Pass', 'p': 'Pass',
    'fail': 'Fail', 'f': 'Fail',
}
df['Result'] = df['Result'].map(result_map)
print(df['Result'].value_counts(dropna=False))
word_to_num = {
    'zero': 0, 'one': 1, 'two': 2, 'three': 3, 'four': 4, 'five': 5,
    'six': 6, 'seven': 7, 'eight': 8, 'nine': 9, 'ten': 10,
    'eleven': 11, 'twelve': 12, 'thirteen': 13, 'fourteen': 14, 'fifteen': 15,
    'sixteen': 16, 'seventeen': 17, 'eighteen': 18, 'nineteen': 19, 'twenty': 20,
}
def to_numeric_safe(series):
    s = series.astype(str).str.strip().str.lower()
    s = s.replace(word_to_num)          # spelled-out numbers
    s = s.replace({'nan': np.nan})
    return pd.to_numeric(s, errors='coerce')   # 'absent', 'error', 'n/a' -> NaN
numeric_like_cols = ['Age', 'Attendance_Percentage', 'Study_Hours_per_day',
                      'Math_Score', 'Science_Score', 'English_Score', 'Total_Score']
for c in numeric_like_cols:
    df[c] = to_numeric_safe(df[c])
df[numeric_like_cols].dtypes
# Age
df.loc[(df['Age'] < 10) | (df['Age'] > 20), 'Age'] = np.nan
# Attendance
df.loc[(df['Attendance_Percentage'] < 0) | (df['Attendance_Percentage'] > 100), 'Attendance_Percentage'] = np.nan
# Study hours
df.loc[(df['Study_Hours_per_day'] < 0) | (df['Study_Hours_per_day'] > 16), 'Study_Hours_per_day'] = np.nan
# Subject scores (0-100)
for c in ['Math_Score', 'Science_Score', 'English_Score']:
    df.loc[(df[c] < 0) | (df[c] > 100), c] = np.nan
df[['Age', 'Attendance_Percentage', 'Study_Hours_per_day',
    'Math_Score', 'Science_Score', 'English_Score']].describe()
# Total_Score should equal Math + Science + English.
# Where all three subject scores are present but Total_Score disagrees (or is
# missing), we recompute it directly instead of treating it as a separate
# unknown — this is more reliable than imputing it statistically.
computed_total = df['Math_Score'] + df['Science_Score'] + df['English_Score']
has_all_three = df[['Math_Score', 'Science_Score', 'English_Score']].notna().all(axis=1)
mismatch = has_all_three & (df['Total_Score'] != computed_total)
print("Rows where Total_Score disagrees with Math+Science+English:", mismatch.sum())
df.loc[has_all_three, 'Total_Score'] = computed_total[has_all_three]
df.loc[(df['Total_Score'] < 0) | (df['Total_Score'] > 300), 'Total_Score'] = np.nan
num_cols = ['Age', 'Attendance_Percentage', 'Study_Hours_per_day',
            'Math_Score', 'Science_Score', 'English_Score', 'Total_Score']
cat_cols = ['Gender', 'City', 'Class', 'Parent_Education', 'Internet_Access']
print("Missing before imputation:")
print(df[num_cols + cat_cols + ['Result']].isna().sum())
# Drop rows with missing target (Result) — can't use a fabricated label
before = df.shape[0]
df = df.dropna(subset=['Result']).reset_index(drop=True)
print(f"Dropped {before - df.shape[0]} rows with missing Result.")
# Numeric -> median imputation
for c in num_cols:
    df[c] = df[c].fillna(df[c].median())
# Categorical -> mode imputation
for c in cat_cols:
    df[c] = df[c].fillna(df[c].mode()[0])
print("Missing after imputation:")
df[num_cols + cat_cols + ['Result']].isna().sum()
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'student_dataset_dirty.csv'

 Part B — Feature Engineering

In [ ]:
df_encoded = df.copy()
# Binary encode Internet_Access (Yes/No -> 1/0)
df_encoded['Internet_Access'] = df_encoded['Internet_Access'].map({'Yes': 1, 'No': 0})
# One-hot encode nominal categorical columns
onehot_cols = ['Gender', 'City', 'Parent_Education']
df_encoded = pd.get_dummies(df_encoded, columns=onehot_cols, drop_first=True)
print("Shape after encoding:", df_encoded.shape)
df_encoded.head()
# 1. Average_Score: mean of the three subject scores
df_encoded['Average_Score'] = df[['Math_Score', 'Science_Score', 'English_Score']].mean(axis=1)
# 2. Study_Efficiency: score gained per hour of daily study (avoids div-by-zero)
df_encoded['Study_Efficiency'] = df_encoded['Average_Score'] / df['Study_Hours_per_day'].replace(0, np.nan)
df_encoded['Study_Efficiency'] = df_encoded['Study_Efficiency'].fillna(df_encoded['Study_Efficiency'].median())
# 3. Attendance_Category: bucket attendance into Low / Medium / High
df_encoded['Attendance_Category'] = pd.cut(
    df['Attendance_Percentage'],
    bins=[0, 60, 85, 100],
    labels=['Low', 'Medium', 'High'],
    include_lowest=True
)
df_encoded = pd.get_dummies(df_encoded, columns=['Attendance_Category'], drop_first=True)

# a low value means consistent performance across subjects.
df_encoded['Score_Consistency'] = df[['Math_Score', 'Science_Score', 'English_Score']].std(axis=1)
df_encoded[['Average_Score', 'Study_Efficiency', 'Score_Consistency']].describe()
numeric_features_to_scale = [
    'Age', 'Attendance_Percentage', 'Study_Hours_per_day',
    'Average_Score', 'Study_Efficiency', 'Score_Consistency'
]
scaler = StandardScaler()
df_scaled_preview = df_encoded.copy()
df_scaled_preview[numeric_features_to_scale] = scaler.fit_transform(df_scaled_preview[numeric_features_to_scale])
df_scaled_preview[numeric_features_to_scale].describe()
corr_features = ['Age', 'Attendance_Percentage', 'Study_Hours_per_day',
                  'Math_Score', 'Science_Score', 'English_Score',
                  'Average_Score', 'Study_Efficiency', 'Score_Consistency',
                  'Total_Score']
corr_matrix = df_encoded[corr_features].corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix — Numeric Features vs Total_Score')
plt.tight_layout()
plt.show()
print("Correlation of each feature with Total_Score:")
print(corr_matrix['Total_Score'].sort_values(ascending=False))


 Part C — Linear Regression (Predict `Total_Score`)

In [ ]:
exclude_cols = ['Student_ID', 'Name', 'Total_Score',
                'Math_Score', 'Science_Score', 'English_Score', 'Average_Score',
                'Result']
feature_cols_linreg = [c for c in df_encoded.columns if c not in exclude_cols]
# keep only numeric/boolean columns (encoding already handled categoricals)
feature_cols_linreg = df_encoded[feature_cols_linreg].select_dtypes(include=[np.number, bool]).columns.tolist()
X = df_encoded[feature_cols_linreg]
y = df_encoded['Total_Score']
print("Features used:", feature_cols_linreg)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
# Scale numeric features (fit on train only, apply to both -> no leakage)
scale_cols = [c for c in numeric_features_to_scale if c in X_train.columns]
scaler_lr = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[scale_cols] = scaler_lr.fit_transform(X_train[scale_cols])
X_test_scaled[scale_cols] = scaler_lr.transform(X_test[scale_cols])
lin_reg = LinearRegression()
lin_reg.fit(X_train_scaled, y_train)
y_pred = lin_reg.predict(X_test_scaled)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"R^2:  {r2:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
plt.figure(figsize=(7, 7))
plt.scatter(y_test, y_pred, alpha=0.4, s=15, edgecolor='none')
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
plt.plot(lims, lims, 'r--', label='Perfect prediction')
plt.xlabel('Actual Total_Score')
plt.ylabel('Predicted Total_Score')
plt.title('Linear Regression — Actual vs Predicted Total_Score')
plt.legend()
plt.tight_layout()
plt.show()
coef_df = pd.DataFrame({
    'Feature': X_train_scaled.columns,
    'Coefficient': lin_reg.coef_
}).sort_values('Coefficient', key=abs, ascending=False)
print(f"Intercept: {lin_reg.intercept_:.3f}")


 Part D — Logistic Regression (Predict `Result`: Pass/Fail)

In [ ]:
print(df['Result'].value_counts())
y_class = df['Result'].map({'Pass': 1, 'Fail': 0})
print()
print(y_class.value_counts())
feature_cols_logreg = [c for c in feature_cols_linreg if c != 'Total_Score']
# For Part D we CAN use subject scores, since predicting Pass/Fail from
# performance is a legitimate (non-circular) question, unlike Part C.
feature_cols_logreg = [c for c in df_encoded.columns
                        if c not in ['Student_ID', 'Name', 'Result']]
feature_cols_logreg = df_encoded[feature_cols_logreg].select_dtypes(include=[np.number, bool]).columns.tolist()
X_cls = df_encoded[feature_cols_logreg]
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_cls, y_class, test_size=0.2, random_state=RANDOM_STATE, stratify=y_class
)
print("Train shape:", X_train_c.shape, "Test shape:", X_test_c.shape)
print("Train class balance:\n", y_train_c.value_counts(normalize=True))
print("Test class balance:\n", y_test_c.value_counts(normalize=True))
# Scale AFTER the split, fit on train only
scale_cols_c = [c for c in numeric_features_to_scale + ['Math_Score', 'Science_Score', 'English_Score']
                if c in X_train_c.columns]
scaler_c = StandardScaler()
X_train_c_scaled = X_train_c.copy()
X_test_c_scaled = X_test_c.copy()
X_train_c_scaled[scale_cols_c] = scaler_c.fit_transform(X_train_c[scale_cols_c])
X_test_c_scaled[scale_cols_c] = scaler_c.transform(X_test_c[scale_cols_c])
log_reg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
log_reg.fit(X_train_c_scaled, y_train_c)
y_pred_c = log_reg.predict(X_test_c_scaled)
y_proba_c = log_reg.predict_proba(X_test_c_scaled)[:, 1]
acc = accuracy_score(y_test_c, y_pred_c)
prec = precision_score(y_test_c, y_pred_c)
rec = recall_score(y_test_c, y_pred_c)
f1 = f1_score(y_test_c, y_pred_c)
auc = roc_auc_score(y_test_c, y_proba_c)
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 score:  {f1:.4f}")
print(f"ROC AUC:   {auc:.4f}")
cm = confusion_matrix(y_test_c, y_pred_c)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Fail', 'Pass'])
fig, ax = plt.subplots(figsize=(5, 5))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
plt.title('Confusion Matrix — Logistic Regression')
plt.tight_layout()
plt.show()
fpr, tpr, _ = roc_curve(y_test_c, y_proba_c)
plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, label=f'ROC curve (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], 'r--', label='Random guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Logistic Regression')
plt.legend()
plt.tight_layout()
plt.show()
class_counts = df['Result'].value_counts()
class_pct = df['Result'].value_counts(normalize=True) * 100
print(class_counts)
print()
print(class_pct.round(2))
plt.figure(figsize=(5, 4))
class_counts.plot(kind='bar', color=['#4C72B0', '#DD8452'])
plt.title('Class Distribution — Result')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
coef_df_c = pd.DataFrame({
    'Feature': X_train_c_scaled.columns,
    'Coefficient (log-odds)': log_reg.coef_[0],
})
coef_df_c['Odds_Ratio'] = np.exp(coef_df_c['Coefficient (log-odds)'])
coef_df_c = coef_df_c.sort_values('Coefficient (log-odds)', key=abs, ascending=False)

print(f"Intercept (log-odds): {log_reg.intercept_[0]:.4f}")
coef_df_c.head(10)